# Brownian Diffusion
We here show how to set up an Analysis object and use it to first fit an artificial vanadium measurement to obtain the resolution. Next, we use the fitted resolution to fit an artificial measurement of a model with diffusion and some elastic scattering. 

We extract and plot the relevant parameters. Finally, we show how to fit directly to the diffusion model.

In the near future, it will be possible to fit the width and area of the Lorentzian to the diffusion model as well.

In [1]:
# Imports
import pooch

from easydynamics.analysis.analysis import Analysis
from easydynamics.experiment import Experiment
from easydynamics.sample_model import BrownianTranslationalDiffusion
from easydynamics.sample_model import ComponentCollection
from easydynamics.sample_model import DeltaFunction
from easydynamics.sample_model import Gaussian
from easydynamics.sample_model import Lorentzian
from easydynamics.sample_model import Polynomial
from easydynamics.sample_model.background_model import BackgroundModel
from easydynamics.sample_model.instrument_model import InstrumentModel
from easydynamics.sample_model.resolution_model import ResolutionModel
from easydynamics.sample_model.sample_model import SampleModel

# Make the plots interactive
%matplotlib widget

We first create an `Experiment` object to contain the data. The data must either be a hdf5 file or a scipp.DataArray; in both cases it must have coordinates `Q` and `energy`. We here use Pooch to download an example vanadium data set.

The data can be rebinned if needed, but we will show how to do that in a different tutorial.

In [2]:
# Load the vanadium data
vanadium_experiment = Experiment('Vanadium')

file_path = pooch.retrieve(
    url='https://github.com/easyscience/dynamics-lib/raw/refs/heads/master/docs/docs/tutorials/data/vanadium_data_example.h5',
    known_hash='16cc1b327c303feeb88fb9dda5390dc4880b62396b1793f98c6fef0b27c7b873',
)


vanadium_experiment.load_hdf5(filename=file_path)

We can visualize the data in multiple ways, relying on plopp: https://scipp.github.io/plopp/

We here show two ways to look at the data: as a 2d colormap with intensity as function of `Q` and `energy`, and as a slicer with intensity as function of `energy` for various `Q`.

If you want $Q$ on the x axis, then set `transpose_axes=True`

In [3]:
vanadium_experiment.plot_data(slicer=False, transpose_axes=False)

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [4]:
vanadium_experiment.plot_data(slicer=True)

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

We now want to fit the vanadium data to determine our resolution. The scattering from vanadium is almost exclusively incoherent elastic, so we model it as a delta function. We do this by creating a `SampleModel` and adding a `DeltaFunction` component to it. The component acts as a template and gets copied to every `Q` when we attach the `SampleModel` to our `Analysis` object. Let's create the `SampleModel`.

We do not give the `DeltaFunction` a `center` value. In this case, the center will be fixed at 0 energy transfer. We set the start value of the area to 1.

In [5]:
delta_function = DeltaFunction(display_name='DeltaFunction', area=1)
sample_model = SampleModel(components=delta_function)

We now want to define our resolution function. We will here model it as a Gaussian. We create a `ComponentCollection` and append the `Gaussian` to it. We can add as many components to our resolution as we like; sometimes you need several Gaussians and other functions to accurately describe the resolution.

We fix the area of the resolution to have value 1. If we did not do this, we would fit both the area of the delta function and of the resolution Gaussian, and the fit would never converge.

We finally insert the components in a `ResolutionModel`

In [6]:
resolution_components = ComponentCollection()
res_gauss = Gaussian(width=0.1, area=1, display_name='Res. Gauss')
res_gauss.area.fixed = True
resolution_components.append_component(res_gauss)
resolution_model = ResolutionModel(components=resolution_components)

The background intensity was not 0, so we also create a background model. We use a `Polynomial` with a single coefficient, i.e. a flat background. We here show how to create the `BackgroundModel` and add the background in a single line. We could of course also add it like we did for the `SampleModel` or first create a `ComponentCollection` like we did for the `ResolutionModel`

In [7]:
background_model = BackgroundModel(components=Polynomial(coefficients=[0.001]))

We combine the resolution abd background model into an `InstrumentModel`. This model also contains a fittable energy offset to account for instrument misalignment. All components are centered at this energy offset.

In [8]:
instrument_model = InstrumentModel(
    resolution_model=resolution_model,
    background_model=background_model,
)

We are now ready to collect everything in an analysis object. We give it a display name, the experiment, the sample model and the instrument model. It will then automatically generate a model for each `Q` using the templates given in the `SampleModel`, `ResolutionModel` and `BackgroundModel`.

In [9]:
vanadium_analysis = Analysis(
    display_name='Vanadium Full Analysis',
    experiment=vanadium_experiment,
    sample_model=sample_model,
    instrument_model=instrument_model,
)

Let us first fit a single Q index and plot the data and model to see how it looks. For this, we use the `independent` fit method and choose an arbitrary Q index

In [10]:
fit_result_independent_single_Q = vanadium_analysis.fit(fit_method='independent', Q_index=5)
vanadium_analysis.plot_data_and_model(Q_index=5)

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

The fit looks good, so let us fit all Q indices independently and plot the results.

In [11]:
fit_result_independent_all_Q = vanadium_analysis.fit(fit_method='independent')
vanadium_analysis.plot_data_and_model()

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

It can be nice to inspect the fit parameters, and sometimes continue working with them. To do this, we can convert them to a scipp dataset.

We can also plot the parameters as a function of `Q` using the `plot_parameters` method.

In [12]:
# Inspect the Parameters as a scipp Dataset
vanadium_analysis.parameters_to_dataset()

<scipp.Dataset>
Dimensions: Sizes[Q:16, ]
Coordinates:
* Q                         float64           [1/Å]  (Q)  [0.1, 0.226667, ..., 1.87333, 2]
Data:
  DeltaFunction area        float64            [meV]  (Q)  [0.522094, 0.528523, ..., 0.537589, 0.532933]  [0.000308233, 0.000335062, ..., 0.000287459, 0.000280073]
  DeltaFunction center      float64            [meV]  (Q)  [0, 0, ..., 0, 0]  [0, 0, ..., 0, 0]
  Polynomial_c0             float64  [dimensionless]  (Q)  [0.0994206, 0.0948234, ..., 0.0976454, 0.100519]  [5.53666e-06, 5.6978e-06, ..., 4.94578e-06, 4.98191e-06]
  Res. Gauss area           float64            [meV]  (Q)  [1, 1, ..., 1, 1]  [0, 0, ..., 0, 0]
  Res. Gauss center         float64            [meV]  (Q)  [0, 0, ..., 0, 0]  [0, 0, ..., 0, 0]
  Res. Gauss width          float64            [meV]  (Q)  [0.102228, 0.0998022, ..., 0.103462, 0.10188]  [9.25122e-06, 8.81463e-06, ..., 8.02343e-06, 7.89726e-06]
  energy_offset             float64            [meV]  (Q)  [0.00135435, -0.000525217, ..., -0.001399, 0.000553493]  [1.33891e-05, 1.37164e-05, ..., 1.20573e-05, 1.16915e-05]

In [13]:
# Plot some of fitted parameters as a function of Q
vanadium_analysis.plot_parameters(names=['DeltaFunction area'])

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [14]:
vanadium_analysis.plot_parameters(names=['Res. Gauss width'])

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [15]:
vanadium_analysis.plot_parameters(names=['energy_offset'])

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

We are now happy with our resolution function and can start looking at the data we want to fit. We first load and inspect the data in the same way as before. 

In [16]:
diffusion_experiment = Experiment('Diffusion')

file_path = pooch.retrieve(
    url='https://github.com/easyscience/dynamics-lib/raw/refs/heads/master/docs/docs/tutorials/data/diffusion_data_example.h5',
    known_hash='5fe846b19aacbda8b8b936eb2e5310d025dc56c25b0b353521e7d6b921f229ab',
)

diffusion_experiment.load_hdf5(filename=file_path)

In [17]:
diffusion_experiment.plot_data(slicer=True, ymax=4)

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

The data seems to have a sharp elastic peak, a quasielastic peak and a non-zero background. We set up the corresponding `SampleModel` and `BackgroundModel` just like before.

In [18]:
delta_function = DeltaFunction(display_name='DeltaFunction', area=0.2)
lorentzian = Lorentzian(display_name='Lorentzian', area=0.5, width=0.3)
component_collection = ComponentCollection(
    components=[delta_function, lorentzian],
)

sample_model = SampleModel(
    components=component_collection,
)

background_model = BackgroundModel(components=Polynomial(coefficients=[0.001]))

We also create a new instrument_model and attach it to our analysis, giving it the resolution model determined in the vanadium analysis. We further fix all parameters in the resolution model and normalize it.

In [19]:
instrument_model = InstrumentModel(
    background_model=background_model,
    resolution_model=vanadium_analysis.instrument_model.resolution_model,
)
instrument_model.resolution_model.fix_all_parameters()
instrument_model.normalize_resolution()

diffusion_analysis = Analysis(
    display_name='Diffusion Analysis',
    experiment=diffusion_experiment,
    sample_model=sample_model,
    instrument_model=instrument_model,
)

We don't want to fit our resolution anymore, so we fix all the parameters in it.

`Analysis` handles the convolution of the `sample_model` with the `resolution_model`. The calculation is analytical when possible, and otherwise numerical. For numerical convolution, it will give warnings if the result might be inaccurate due to various numerical errors. In that case, two settings can be varied to improve the result.
`upsample_factor` improves accuracy for narrow signals. The default is 5.
`extension_factor` improves accuracy for broad signals. Here, the default is 0.2.
It is furthermore possible to toggle whether detailed balance correction is normalized or not (see next tutorial)

In [20]:
diffusion_analysis.convolution_settings.upsample_factor = 6
diffusion_analysis.convolution_settings.extension_factor = 0.2
diffusion_analysis.convolution_settings.normalize_detailed_balance = True

Before we start fitting it is a good idea to check how good our start guesses are. We do this by plotting the data and the model using `plot_data_and_model`:

In [21]:
diffusion_analysis.plot_data_and_model()

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

The start guesses are not perfect, but they look good enough to start fitting. Let's give it a try!

In [22]:
diffusion_analysis.fit(fit_method='independent')
diffusion_analysis.plot_data_and_model()

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

The fit looks good, so now we want to look at the most interesting fit parameters: the width and area of the Lorentzian. In later versions of EasyDynamics it will be possible to fit them to e.g. a DiffusionModel.

In [23]:
# Let us look at the most interesting fit parameters
diffusion_analysis.plot_parameters(names=['Lorentzian width', 'Lorentzian area'])

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

There is a clear trend: the area is more or less constant, while the width seems to increase with `Q^2`. We therefore try and fit a Brownian translational diffusion model to the data. In this model, the scattering is given by
$$
I(Q,E) = S \frac{\Gamma(Q)}{\Gamma(Q)^2 + E^2},
$$
where $\Gamma(Q) = D Q^2$ and $D$ is the diffusion coefficient. $S$ is an overall scale.

In addition to this diffusion model, there is still the elastic incoherent scattering.

We create a new `SampleModel` which as a `DeltaFunction` component for the elastic incoherent scattering and a `BrownianTranslationalDiffusion` diffusion model to describe the rest. We also create a new `BackgroundModel` and `InstrumentModel`.

In [24]:
delta_function = DeltaFunction(display_name='DeltaFunction', area=0.2)
component_collection = ComponentCollection(
    components=[delta_function],
)
diffusion_model = BrownianTranslationalDiffusion(
    display_name='Brownian Translational Diffusion', diffusion_coefficient=2.4e-9, scale=0.5
)

sample_model = SampleModel(
    components=component_collection,
    diffusion_models=diffusion_model,
)

background_model = BackgroundModel(components=Polynomial(coefficients=[0.001]))

In [25]:
instrument_model = InstrumentModel(
    background_model=background_model,
    resolution_model=vanadium_analysis.instrument_model.resolution_model,
)

We attach all our models to a new `Analysis` object.

In [26]:
diffusion_model_analysis = Analysis(
    display_name='Diffusion Full Analysis',
    experiment=diffusion_experiment,
    sample_model=sample_model,
    instrument_model=instrument_model,
)

diffusion_model_analysis.instrument_model.resolution_model.fix_all_parameters()

As always, we first check the start parameters before we fit

In [27]:
diffusion_model_analysis.plot_data_and_model()

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

We can now fit all the data simultaneously to our model. It looks good!

In [28]:
diffusion_model_analysis.fit(fit_method='simultaneous')
diffusion_model_analysis.plot_data_and_model(ymax=2)

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

It does not make sense to plot the diffusion parameters, but we can display them (with uncertainties) like this.

In [29]:
diffusion_model.get_all_parameters()

[<Parameter 'diffusion_coefficient': 1.126e-08 ± 9.799e-11 m^2/s, bounds=[0.0:inf]>,
 <Parameter 'scale': 0.6938 ± 0.0043 meV, bounds=[0.0:inf]>]